# Notebook 1 - Data Collection & Portfolio Construction
This notebook downloads market data, validates it, constructs a weighted portfolio and exports clean datasets.

In [1]:
# Install once if required
# !pip install yfinance pandas numpy openpyxl

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

START_DATE="2020-01-01"
END_DATE="2026-07-01"

PORTFOLIO={
    "RELIANCE.NS":0.20,
    "HDFCBANK.NS":0.15,
    "TCS.NS":0.15,
    "INFY.NS":0.10,
    "ICICIBANK.NS":0.10,
    "BHARTIARTL.NS":0.10,
    "ITC.NS":0.10,
    "TATAMOTORS.NS":0.10
}

Path("data/raw").mkdir(parents=True,exist_ok=True)
Path("data/processed").mkdir(parents=True,exist_ok=True)


In [3]:
stocks=list(PORTFOLIO.keys())

prices=yf.download(
    stocks,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)["Close"]

nifty=yf.download(
    "^NSEI",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)["Close"].rename("NIFTY")

vix=yf.download(
    "^INDIAVIX",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)["Close"].rename("VIX")

prices=prices.sort_index().ffill().dropna(how="all")
nifty=nifty.ffill()
vix=vix.ffill()


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


TypeError: 'str' object is not callable

In [ ]:
returns=prices.pct_change().dropna()

weights=pd.Series(PORTFOLIO)
weights=weights.reindex(returns.columns)

portfolio_return=returns.dot(weights)

portfolio=pd.DataFrame(index=portfolio_return.index)
portfolio["Portfolio_Return"]=portfolio_return
portfolio["Portfolio_Growth"]=(1+portfolio_return).cumprod()
portfolio["Portfolio_Value"]=1000000*portfolio["Portfolio_Growth"]

portfolio=portfolio.join(nifty).join(vix)

portfolio.to_csv("data/processed/portfolio_daily.csv")
returns.to_csv("data/processed/stock_returns.csv")

portfolio.head()


## Next Notebook

Notebook 2 will calculate:

- CAGR
- Annual Return
- Volatility
- Sharpe Ratio
- Sortino Ratio
- Beta
- Alpha
- Maximum Drawdown
- VaR
- CVaR
- Correlation Matrix
- Risk Attribution
